# Fine-tuning ModernBERT on GoEmotions with LoRA Adapters
**Task:** Multi-label emotion classification (28 classes)  
**Model:** `answerdotai/ModernBERT-base`  
**Method:** LoRA Adapter fine-tuning (`LoRAConfig`)

In [ ]:
# Install / upgrade all required packages
!pip install -q transformers datasets accelerate kagglehub scikit-learn seaborn adapters

In [ ]:
# Confirm GPU is available
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Core imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, f1_score, hamming_loss

from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
import kagglehub

In [ ]:
# ── All hyperparameters in one place ─────────────────────────────────────────────
MODEL_NAME  = "answerdotai/ModernBERT-base"
MAX_LENGTH  = 128       # GoEmotions comments are short, 128 is sufficient
BATCH_SIZE  = 32        # reduce to 16 if you get OOM errors
EPOCHS      = 5
LR          = 2e-5
THRESHOLD   = 0.5       # sigmoid cutoff for predicting a label as active
OUTPUT_DIR  = "./modernbert-goemotions-lora"
SEED        = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ── Download dataset from Kaggle and locate the data files ───────────────────
data_path = kagglehub.dataset_download("debarshichanda/goemotions")
print("Dataset root:", data_path)

# The TSV files live inside a 'data/' subdirectory
tsv_dir = os.path.join(data_path, "data")
print("Files in data/:", os.listdir(tsv_dir))

# TSV files have no header; columns are: text | label_ids | comment_id
def load_split(filename):
    df = pd.read_csv(
        os.path.join(tsv_dir, filename),
        sep="\t", header=None,
        names=["text", "labels", "comment_id"]
    )
    df["labels"] = df["labels"].apply(lambda x: [int(i) for i in str(x).split(",")])
    return df

train_df = load_split("train.tsv")
val_df   = load_split("dev.tsv")
test_df  = load_split("test.tsv")

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
train_df.head(3)

In [ ]:
# ── Load the 28 emotion class names ──────────────────────────────────────────────
with open(os.path.join(tsv_dir, "emotions.txt")) as f:
    emotion_labels = [line.strip() for line in f]

NUM_LABELS = len(emotion_labels)
print(f"{NUM_LABELS} classes:", emotion_labels)

In [ ]:
# ── Quick EDA — label frequency + average labels per sample ───────────────────
counts = np.zeros(NUM_LABELS, dtype=int)
for row in train_df["labels"]:
    for idx in row:
        counts[idx] += 1

freq = pd.Series(counts, index=emotion_labels).sort_values(ascending=False)

plt.figure(figsize=(15, 4))
sns.barplot(x=freq.index, y=freq.values, palette="viridis")
plt.xticks(rotation=45, ha="right")
plt.title("GoEmotions — training label frequency")
plt.tight_layout()
plt.show()

avg = train_df["labels"].apply(len).mean()
print(f"Avg labels per sample: {avg:.2f}")

In [ ]:
# ── Convert label-index lists to multi-hot float vectors ─────────────────────
def to_multi_hot(label_lists, n):
    mat = np.zeros((len(label_lists), n), dtype=np.float32)
    for i, lbls in enumerate(label_lists):
        for l in lbls:
            mat[i, l] = 1.0
    return mat

train_labels = to_multi_hot(train_df["labels"], NUM_LABELS)
val_labels   = to_multi_hot(val_df["labels"],   NUM_LABELS)
test_labels  = to_multi_hot(test_df["labels"],  NUM_LABELS)

print("Label matrix shape (train):", train_labels.shape)

In [ ]:
# ── Tokenise all splits — padding is handled dynamically by the data collator ─
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(list(texts), truncation=True, max_length=MAX_LENGTH, padding=False)

train_enc = tokenize(train_df["text"])
val_enc   = tokenize(val_df["text"])
test_enc  = tokenize(test_df["text"])
print("Tokenisation complete.")

In [ ]:
# ── PyTorch Dataset wrapper ────────────────────────────────────────────────
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = EmotionDataset(train_enc, train_labels)
val_dataset   = EmotionDataset(val_enc,   val_labels)
test_dataset  = EmotionDataset(test_enc,  test_labels)
print(f"Dataset sizes — train: {len(train_dataset):,}, val: {len(val_dataset):,}, test: {len(test_dataset):,}")

In [ ]:
# ── Load ModernBERT with a 28-class head + LoRA Adapter ──────────────────────
import adapters
from adapters import LoRAConfig

print(f"Loading model: {MODEL_NAME}")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
)

# Initialise adapter support on the loaded model
adapters.init(model)

# Add LoRA adapter
config = LoRAConfig(r=8, alpha=16)
model.add_adapter("lora_adapter", config=config)

# Freeze backbone; train only the adapter weights
model.train_adapter("lora_adapter")

# Also unfreeze the classification head (randomly initialised, needs training)
for name, param in model.named_parameters():
    if "classifier" in name or "pooler" in name:
        param.requires_grad = True

model.to(DEVICE)

total     = sum(p.numel() for p in model.parameters()) / 1e6
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6
print(f"Total params: {total:.1f}M  |  Trainable: {trainable:.1f}M ({trainable/total*100:.1f}%)")

In [ ]:
# ── Metrics reported after each epoch ──────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= THRESHOLD).astype(int)
    return {
        "micro_f1":     f1_score(labels, preds, average="micro",  zero_division=0),
        "macro_f1":     f1_score(labels, preds, average="macro",  zero_division=0),
        "hamming_loss": hamming_loss(labels, preds),
    }

In [ ]:
# ── Training configuration ───────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "micro_f1",
    greater_is_better           = True,
    logging_steps               = 100,
    fp16                        = True,
    seed                        = SEED,
    report_to                   = "none",
)

In [ ]:
# ── Build the Trainer and start fine-tuning ──────────────────────────────────
trainer = Trainer(
    model              = model,
    args               = training_args,
    train_dataset      = train_dataset,
    eval_dataset       = val_dataset,
    processing_class   = tokenizer,
    data_collator      = DataCollatorWithPadding(tokenizer),
    compute_metrics    = compute_metrics,
    callbacks          = [EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
# ── Full evaluation on the held-out test set ──────────────────────────────
predictions = trainer.predict(test_dataset)
probs = torch.sigmoid(torch.tensor(predictions.predictions)).numpy()
preds = (probs >= THRESHOLD).astype(int)

print(classification_report(test_labels, preds, target_names=emotion_labels, zero_division=0))
print(f"Micro-F1     : {f1_score(test_labels, preds, average='micro',  zero_division=0):.4f}")
print(f"Macro-F1     : {f1_score(test_labels, preds, average='macro',  zero_division=0):.4f}")
print(f"Hamming Loss : {hamming_loss(test_labels, preds):.4f}")

In [ ]:
# ── Per-class F1 bar chart ─────────────────────────────────────────────────
report = classification_report(
    test_labels, preds, target_names=emotion_labels,
    zero_division=0, output_dict=True
)
f1_series = pd.Series(
    {lbl: report[lbl]["f1-score"] for lbl in emotion_labels}
).sort_values(ascending=False)

plt.figure(figsize=(15, 4))
sns.barplot(x=f1_series.index, y=f1_series.values, palette="magma")
plt.xticks(rotation=45, ha="right")
plt.title("Per-class F1 — ModernBERT + LoRA Adapter on GoEmotions (test set)")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}/")